# Residual U-Net (ResUNet) -- fixed 592x592

Trains a **ResUNet** for 3-class (Background / Artery / Vein) segmentation under the
exact same protocol as every other fixed-592 single-stage model in this project:

- Fixed 592x592 resize (bilinear for images, nearest for masks)
- AdamW (lr=5e-4, weight_decay=2e-4) + CosineAnnealingLR (T_max=300)
- CEDiceLoss (0.7 x weighted-CE + 0.3 x SoftDice on vessel classes)
- Batch size 4, up to 300 epochs, early stopping (patience 20 eval checks = 100 epochs)
- Augmentation: h-flip, v-flip, rotation +-10 deg

## Architecture

ResUNet replaces the plain DoubleConv blocks of Standard U-Net with **residual blocks**:
each block adds a learned identity shortcut (1x1 conv + BN when channel count changes,
identity otherwise) around the two 3x3 conv layers, giving the network a direct gradient
path through every encoder and decoder stage. Channel progression is 64->128->256->512->1024
(identical to Standard U-Net and BiFPN-UNet 64-1024ch), so the comparison is a clean
ablation of residual connections vs. plain convolutions at the same capacity.

## Imports

In [ ]:
import os
import copy
import random
import gc
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as T

print("All imports successful.")


## Constants

In [ ]:
NUM_CLASSES    = 3
CLASS_NAMES    = {0: 'Background', 1: 'Artery', 2: 'Vein'}
VESSEL_CLASSES = [1, 2]


## Dataset -- identical to every other fixed-592 notebook

In [ ]:
class ImageDataset(data.Dataset):
    def __init__(self, root, new_size=592, supervised=True, augment=False):
        self.root = Path(root)
        self.new_size = new_size
        self.supervised = supervised
        self.augment = augment

        img_dir = self.root / 'images'
        ann_dir = self.root / 'annotation'
        self.image_paths = sorted(img_dir.glob('*.png'))
        self.label_paths = [ann_dir / f'{p.stem}.png' for p in self.image_paths] if supervised else []

        print(f"Dataset root        : {root}")
        print(f"  Images found      : {len(self.image_paths)}")
        if supervised:
            print(f"  Annotations found : {len(self.label_paths)}")
        print(f"  Augmentation      : {self.augment}")
        if self.image_paths:
            print(f"  First 5 images    : {[p.name for p in self.image_paths[:5]]}")

    @staticmethod
    def rgb_to_class_mask(rgb):
        r, g, b = rgb[:, :, 0], rgb[:, :, 1], rgb[:, :, 2]
        mask = np.zeros(rgb.shape[:2], dtype=np.int64)
        mask[(r > 180) & (g < 80)  & (b < 80)]  = 1
        mask[(r < 80)  & (g < 80)  & (b > 180)] = 2
        return mask

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, index):
        s = self.new_size
        image = Image.open(self.image_paths[index]).convert('RGB').resize((s, s), Image.BILINEAR)
        if not self.supervised:
            return T.ToTensor()(image)
        ann = Image.open(self.label_paths[index]).convert('RGB').resize((s, s), Image.NEAREST)
        if self.augment:
            if random.random() < 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)
                ann   = ann.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() < 0.5:
                image = image.transpose(Image.FLIP_TOP_BOTTOM)
                ann   = ann.transpose(Image.FLIP_TOP_BOTTOM)
            angle = random.uniform(-10, 10)
            image = image.rotate(angle, resample=Image.BILINEAR,  fillcolor=(0, 0, 0))
            ann   = ann.rotate(angle,   resample=Image.NEAREST, fillcolor=(0, 0, 0))
        return T.ToTensor()(image), torch.from_numpy(self.rgb_to_class_mask(np.array(ann)))


## Load data

In [ ]:
root = '/home/mefeki/Bureau/work/Fundus-AVSeg-prep-data'

test_dataset  = ImageDataset(root=os.path.join(root, 'test'),  augment=False)
test_loader   = data.DataLoader(test_dataset, shuffle=False, batch_size=1)
print(f"Test images: {len(test_dataset)}")


## ResUNet Architecture

Each encoder and decoder stage uses a **ResidualBlock** instead of the plain DoubleConv
of Standard U-Net. A 1x1 projection shortcut is added whenever the input and output
channel counts differ (first conv of each stage); all other shortcuts are the identity.
Everything else -- max-pool downsampling, transposed-conv upsampling, skip concatenation,
final 1x1 classification head -- is identical to Standard U-Net.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.shortcut = (
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch))
            if in_ch != out_ch else nn.Identity()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.block(x) + self.shortcut(x))


class ResDown(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool  = nn.MaxPool2d(2)
        self.block = ResidualBlock(in_ch, out_ch)

    def forward(self, x):
        return self.block(self.pool(x))


class ResUp(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up    = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.block = ResidualBlock(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        if x1.shape[-2:] != x2.shape[-2:]:
            x1 = F.interpolate(x1, size=x2.shape[-2:], mode='bilinear', align_corners=False)
        return self.block(torch.cat([x2, x1], dim=1))


class ResUNet(nn.Module):
    def __init__(self, in_channels=3, n_classes=3):
        super().__init__()
        # Encoder (same 64->1024 channel progression as Standard U-Net)
        self.inc   = ResidualBlock(in_channels, 64)
        self.down1 = ResDown(64,   128)
        self.down2 = ResDown(128,  256)
        self.down3 = ResDown(256,  512)
        self.down4 = ResDown(512, 1024)
        # Decoder
        self.up1   = ResUp(1024,  512)
        self.up2   = ResUp(512,   256)
        self.up3   = ResUp(256,   128)
        self.up4   = ResUp(128,    64)
        self.outc  = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x  = self.up1(x5, x4)
        x  = self.up2(x,  x3)
        x  = self.up3(x,  x2)
        x  = self.up4(x,  x1)
        return self.outc(x)


## Loss: Weighted CE + Soft Dice (same as all other fixed-592 models)

In [ ]:
class SoftDiceLoss(nn.Module):
    def __init__(self, classes=VESSEL_CLASSES, eps=1e-6):
        super().__init__()
        self.classes = classes
        self.eps = eps

    def forward(self, logits, labels):
        probs = torch.softmax(logits, dim=1)
        losses = []
        for c in self.classes:
            p = probs[:, c]; t = (labels == c).float()
            tp = (p * t).sum(dim=(1, 2))
            fp = (p * (1 - t)).sum(dim=(1, 2))
            fn = ((1 - p) * t).sum(dim=(1, 2))
            losses.append((1 - (2*tp + self.eps) / (2*tp + fp + fn + self.eps)).mean())
        return torch.stack(losses).mean()


class CEDiceLoss(nn.Module):
    def __init__(self, class_weights=None, dice_weight=0.3):
        super().__init__()
        self.ce   = nn.CrossEntropyLoss(weight=class_weights)
        self.dice = SoftDiceLoss()
        self.dice_weight = dice_weight

    def forward(self, logits, labels):
        return (1 - self.dice_weight) * self.ce(logits, labels.long()) + \
               self.dice_weight * self.dice(logits, labels.long())


## Metrics

In [ ]:
def dice_coef_per_class(preds, labels, num_classes=NUM_CLASSES, eps=1e-6):
    per_class = {}
    for c in range(num_classes):
        pred_c  = (preds  == c).float()
        label_c = (labels == c).float()
        tp = float(torch.sum(pred_c * label_c))
        fp = float(torch.sum(pred_c * (1 - label_c)))
        fn = float(torch.sum((1 - pred_c) * label_c))
        per_class[c] = 2 * tp / (2 * tp + fp + fn + eps)
    return per_class, float(np.mean([per_class[c] for c in VESSEL_CLASSES]))


## Training & test functions (crash-safe checkpointing, early stopping)

In [ ]:
def test_model(model, dataloader, verbose=True):
    model.eval()
    total_dice, n = 0.0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            preds = torch.argmax(torch.softmax(model(inputs), dim=1), dim=1)
            _, vessel_dice = dice_coef_per_class(preds, labels)
            total_dice += vessel_dice
            n += 1
    final_dice = total_dice / max(n, 1)
    if verbose:
        print(f'\nMean vessel Dice on test set: {final_dice:.4f}')
    return final_dice


def train_model(model, criterion, optimizer, scheduler, train_loader, test_loader,
                num_epochs, eval_every=5,
                checkpoint_path='models/best_model_resunet_592.pth',
                history_path='training_history_resunet_592.csv',
                patience_evals=10):
    best_dice, best_state = -1.0, copy.deepcopy(model.state_dict())
    epoch_losses, eval_history = [], []
    evals_since_best = 0

    os.makedirs(os.path.dirname(checkpoint_path) or '.', exist_ok=True)
    with open(history_path, 'w') as f:
        f.write('epoch,test_vessel_dice,is_best\n')

    for epoch in range(num_epochs):
        model.train()
        batch_losses = []
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        scheduler.step()
        mean_loss = sum(batch_losses) / len(batch_losses)
        epoch_losses.append(mean_loss)

        is_last = (epoch == num_epochs - 1)
        if (epoch + 1) % eval_every == 0 or is_last:
            dice = test_model(model, test_loader, verbose=False)
            eval_history.append((epoch + 1, dice))
            is_best = dice > best_dice
            if is_best:
                best_dice  = dice
                best_state = copy.deepcopy(model.state_dict())
                torch.save(best_state, checkpoint_path)
                evals_since_best = 0
            else:
                evals_since_best += 1
            with open(history_path, 'a') as f:
                f.write(f'{epoch+1},{dice:.6f},{int(is_best)}\n')
            print(f'Epoch {epoch+1}/{num_epochs}  Loss: {mean_loss:.4f}  '
                  f'Test vessel Dice: {dice:.4f}  (best so far: {best_dice:.4f})')
            if patience_evals is not None and evals_since_best >= patience_evals:
                print(f'Early stopping: no improvement in {patience_evals} eval '
                      f'checks ({patience_evals * eval_every} epochs).')
                break
        else:
            print(f'Epoch {epoch+1}/{num_epochs}  Loss: {mean_loss:.4f}')

    print('Training complete.')
    model.load_state_dict(best_state)
    return model, epoch_losses, eval_history, best_dice


## Setup

In [ ]:
gc.collect()
torch.cuda.empty_cache()


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

train_root = os.path.join(root, 'train')
train_dataset_full = ImageDataset(root=train_root, supervised=True, augment=True)
print(f"Total training images: {len(train_dataset_full)}")


## Training

In [ ]:
EPOCHS         = 300
BATCH_SIZE     = 4
SEED           = 42
EVAL_EVERY     = 5
DICE_WEIGHT    = 0.3
PATIENCE_EVALS = 20

full_dataloader = data.DataLoader(
    train_dataset_full, shuffle=True, batch_size=BATCH_SIZE,
    num_workers=4, pin_memory=True, persistent_workers=True)

print(f'\n=== Training on full training set ({len(train_dataset_full)} images) ===')

torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

# Class weights
pixel_counts = Counter()
plain_ds = ImageDataset(root=train_root, supervised=True, augment=False)
for _, lbl in plain_ds:
    for c in range(NUM_CLASSES):
        pixel_counts[c] += int((lbl == c).sum())
total = sum(pixel_counts.values())
freqs = torch.tensor([pixel_counts[c] / total for c in range(NUM_CLASSES)])
inv_freq = 1.0 / (freqs + 1e-6)
class_weights = (inv_freq / inv_freq[1:].mean()).to(device)
print("Class pixel %:", {CLASS_NAMES[c]: f"{100*freqs[c].item():.2f}%" for c in range(NUM_CLASSES)})
print("Class weights:", class_weights)

model     = ResUNet(in_channels=3, n_classes=NUM_CLASSES).to(device)
criterion = CEDiceLoss(class_weights, dice_weight=DICE_WEIGHT)
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=2e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

save_path = 'models/best_model_resunet_592.pth'
model, training_losses, eval_history, best_dice = train_model(
    model, criterion, optimizer, scheduler,
    full_dataloader, test_loader,
    num_epochs=EPOCHS, eval_every=EVAL_EVERY,
    checkpoint_path=save_path,
    history_path='training_history_resunet_592.csv',
    patience_evals=PATIENCE_EVALS)

final_dice = test_model(model, test_loader)
print(f"\nBest test vessel Dice during training        : {best_dice:.4f}")
print(f"Final-epoch test vessel Dice (no selection)   : {eval_history[-1][1]:.4f}")
print(f"Standard U-Net (RGB, 592x592) baseline        : 0.8023")
print(f"BiFPN-UNet (64-1024ch, 300 epochs)            : 0.8092")
print(f"Morpho-U-Net (TopHat+Dist, plain backbone)    : 0.7917")
print(f"BiFPN-UNet (lean 32-256ch)                    : 0.7906")
print(f"{'Beats' if best_dice > 0.8023 else 'Does not beat'} the Standard U-Net baseline.")
print(f"Model saved to {save_path}")

fig, ax1 = plt.subplots(figsize=(8, 6))
ax1.plot(range(1, len(training_losses)+1), training_losses, color='tab:blue', label='Train Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Train Loss', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax2 = ax1.twinx()
eval_epochs, eval_dices = zip(*eval_history)
ax2.plot(eval_epochs, eval_dices, color='tab:orange', marker='o', label='Test Vessel Dice')
ax2.axhline(0.8023, color='gray', linestyle='--', label='Standard U-Net baseline (0.8023)')
ax2.set_ylabel('Test Vessel Dice', color='tab:orange')
ax2.tick_params(axis='y', labelcolor='tab:orange')
fig.legend(loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=3)
plt.title(f'ResUNet (fixed 592x592) | Best Dice: {best_dice:.4f}')
fig.tight_layout()
plt.savefig('training_diagnostics_resunet_592.png', dpi=150, bbox_inches='tight')
plt.show()


## Per-class breakdown

In [ ]:
model.eval()
with torch.no_grad():
    print(f"{'Class':<15} {'Dice':>8}")
    print('-' * 25)
    all_class_dice = {c: [] for c in range(NUM_CLASSES)}
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        preds = torch.argmax(torch.softmax(model(inputs), dim=1), dim=1)
        per_class, _ = dice_coef_per_class(preds, labels)
        for c, d in per_class.items():
            all_class_dice[c].append(d)
    for c in range(NUM_CLASSES):
        print(f"  {CLASS_NAMES[c]:<13} {np.mean(all_class_dice[c]):>8.4f}")
    mean_vessel = np.mean([np.mean(all_class_dice[c]) for c in VESSEL_CLASSES])
    print(f"\n  Mean vessel Dice (artery + vein): {mean_vessel:.4f}")
